In [ ]:
# OLD MODELS

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv("Dataset/Features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Function to create specific targets (Will I do X in the next 30 mins?)
def get_target(df, activity_name):
    times = df[df['activity'] == activity_name]['timestamp'].sort_values()
    df[f'target_{activity_name}'] = df['timestamp'].apply(
        lambda x: 1 if any((t > x and (t - x).total_seconds()/60 <= 30) for t in times) else 0
    )
    return df[f'target_{activity_name}']

# Define activities and features
activities = ['drink', 'eat', 'outside_out']
features = ["hour", "day_of_week", "is_weekend", "minutes_since_midnight", 
            "time_since_last_drink", "time_since_last_eat", "time_since_last_outside"]

print("--- PERFORMANCE RESULTS ---")

for act in activities:
    y = get_target(df, act)
    X = df[features].fillna(0)
    
    # Split 80% training, 20% testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model.fit(X_train, y_train)
    
    print(f"\nModel: {act.upper()}")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

--- PERFORMANCE RESULTS ---

Model: DRINK
              precision    recall  f1-score   support

           0       1.00      0.94      0.97        16
           1       0.75      1.00      0.86         3

    accuracy                           0.95        19
   macro avg       0.88      0.97      0.91        19
weighted avg       0.96      0.95      0.95        19


Model: EAT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19


Model: OUTSIDE_OUT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19



In [4]:
import pandas as pd

# Define your scenarios with the new '7-day average' columns
# Scenario 1: Monday - Drank 8 times today, average for the week is 7
# Scenario 2: Friday - Drank 6 times today, average for the week is 6.5
# Scenario 3: Saturday - High activity today (10), but average is lower (8)
# Scenario 4: Sunday - Low activity today (5), average is 7.5
data = {
    'day_of_week': [0, 4, 5, 6],
    'is_weekend': [0, 0, 1, 1],
    'drink': [8, 6, 10, 5],
    'eat': [3, 2, 4, 2],
    'drink_7day_avg': [7.0, 6.5, 8.0, 7.5],
    'eat_7day_avg': [3.0, 2.5, 3.5, 3.0]
}

# Create the DataFrame
df_test_daily = pd.DataFrame(data)

# Save it to the Dataset folder
import os
if not os.path.exists('Dataset'):
    os.makedirs('Dataset')

df_test_daily.to_csv("Dataset/Daily_Test_Data.csv", index=False)

print("Daily_Test_Data.csv successfully updated with 7-day averages!")
print(df_test_daily)

Daily_Test_Data.csv successfully updated with 7-day averages!
   day_of_week  is_weekend  drink  eat  drink_7day_avg  eat_7day_avg
0            0           0      8    3             7.0           3.0
1            4           0      6    2             6.5           2.5
2            5           1     10    4             8.0           3.5
3            6           1      5    2             7.5           3.0


In [7]:
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error

# Load mode and test Data
model_drink = joblib.load("daily_drink_forecast.roboai")
test_data = pd.read_csv("Dataset/Daily_Test_Data.csv")

# Make Predictions (model uses today's activity to predict tomorrow's total)
predictions = model_drink.predict(test_data[['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg']])

# Display Results
test_data['predicted_drinks_tomorrow'] = predictions.round(1)

print("--- DRINKING FORECAST TEST ---")
print(test_data[['day_of_week', 'drink', 'predicted_drinks_tomorrow']])

--- DRINKING FORECAST TEST ---
   day_of_week  drink  predicted_drinks_tomorrow
0            0      8                        6.6
1            4      6                        6.0
2            5     10                        4.6
3            6      5                        4.4


In [8]:
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error

# Load the model and test Data
model_eat = joblib.load("daily_eat_forecast.roboai")
test_data = pd.read_csv("Dataset/Daily_Test_Data.csv")

# Make Predictions
predictions = model_eat.predict(test_data[['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg']])

# Display Results
test_data['predicted_meals_tomorrow'] = predictions.round(1)

print("--- EATING FORECAST TEST ---")
print(test_data[['day_of_week', 'eat', 'predicted_meals_tomorrow']])

--- EATING FORECAST TEST ---
   day_of_week  eat  predicted_meals_tomorrow
0            0    3                       3.6
1            4    2                       2.2
2            5    4                       2.5
3            6    2                       2.2
